# 重建可视化（z_sem 复活判据）
**回答的问题：z_sem 到底携带了多少图像内容。**

四列：原图 | DDIM 反演 | 反演+重建 | 仅 z_sem 生成（共享噪声）。
第四列全糊 = z_sem 没携带信息；能认出对应图 = 携带了。
cond 走训练同分布路径（归一化原图过 backbone），与训练条件一致。

In [ ]:
import os, sys
import torch

# ---- 路径：notebook 在 visual/ 下，切到 mmsegmentation 作为工作目录 ----
MMSEG = os.path.abspath('../tasks/Segmentation/mmsegmentation')
os.chdir(MMSEG)
sys_path_added = []
for p in [MMSEG, os.path.join(MMSEG, 'eden')]:
    import sys
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin, eden.pcgrad_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)
sys.modules.setdefault('mydiffae.pcgrad_mixin', eden.pcgrad_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

sys.path.insert(0, os.path.abspath('../visual'))
import probe_core as pc


In [ ]:
CFG = 'configs/fcn/fcn_r50-d8_4xb4-80k_ade20k-512x512_diffae_mask.py'
CKPT = 'work_dirs/eden/fcn_diffae_mask/iter_80000.pth'
OUT = 'work_dirs/offline_probe/recon'
NUM_IMAGES = 4
os.makedirs(OUT, exist_ok=True)

In [ ]:
model, cfg = pc.build_model(CFG, CKPT)
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
path = pc.save_recon(model, batch, os.path.join(OUT, 'recon.png'), NUM_IMAGES)
print(path)

In [ ]:
from PIL import Image
Image.open(path)